In [0]:
dbutils.widgets.text("catalog", "fk_ce1366")
catalog = dbutils.widgets.get("catalog")

In [0]:
from pyspark.sql.functions import *

In [0]:
df = spark.read.table(f"{catalog}.bronze.customers")

#casting & data quality
df = (
    df.filter(col("customer_id").isNotNull())
    .withColumn("full_name", trim(initcap(col("full_name"))))
    .withColumn("email", trim(lower(col("email"))))
    .withColumn("phone", trim(regexp_replace(col("phone"), "[^0-9+]", "")))
    .withColumn("city", trim(initcap(col("city"))))
    .withColumn("tier", trim(initcap(col("tier"))))
    .withColumn("signup_date", try_to_date(col("signup_date"),  "yyyy-MM-dd"))
)

df.write.mode("overwrite").saveAsTable(f"{catalog}.silver.customers")


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalog}.silver.customers_scd_t2 (
    customer_id STRING,
    full_name STRING,
    email STRING,
    phone STRING,
    city STRING,
    tier STRING,
    effective_from DATE,
    effective_to DATE,
    is_current BOOLEAN
)
USING DELTA          
""")


DataFrame[]

In [0]:
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW snapshot_source AS
SELECT
    customer_id,
    full_name,
    email,
    phone,
    city,
    tier,
    DATE '2026-03-01' AS effective_from,
    CAST(NULL AS DATE) AS effective_to,
    TRUE AS is_current
FROM (
    SELECT DISTINCT
        customer_id, full_name, email, phone, city, tier
    FROM {catalog}.silver.customers
) s
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY customer_id
    ORDER BY customer_id
) = 1         
""")

DataFrame[]

In [0]:
spark.sql(f"""
MERGE INTO {catalog}.silver.customers_scd_t2 AS t
USING snapshot_source AS s
ON t.customer_id = s.customer_id
WHEN NOT MATCHED THEN
    INSERT (
        customer_id, full_name, email, phone, city, tier,
        effective_from, effective_to, is_current
    )
    VALUES (
        s.customer_id, s.full_name, s.email, s.phone,
        s.city, s.tier, s.effective_from,
        s.effective_to, s.is_current
    ) 
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
target = f"{catalog}.silver.customers_scd_t2"
source = f"{catalog}.silver.customers"

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW type2_changes AS
SELECT s.customer_id, s.full_name, s.email, s.phone, s.city, s.tier, s.signup_date
FROM (
    SELECT DISTINCT customer_id, full_name, email, phone, city, tier, signup_date
    FROM {source}
    WHERE customer_id IS NOT NULL AND signup_date IS NOT NULL
) s
JOIN {target} t
    ON s.customer_id = t.customer_id AND t.is_current = TRUE
WHERE s.signup_date > t.effective_from
  AND (NOT (s.city <=> t.city) OR NOT (s.tier <=> t.tier))
""")

spark.sql(f"""
INSERT INTO {target}
SELECT
    s.customer_id, s.full_name, s.email, s.phone, s.city, s.tier,
    s.signup_date AS effective_from, NULL AS effective_to, TRUE AS is_current
FROM (
    SELECT DISTINCT customer_id, full_name, email, phone, city, tier, signup_date
    FROM {source}
    WHERE customer_id IS NOT NULL AND signup_date IS NOT NULL
) s
LEFT JOIN {target} t
    ON s.customer_id = t.customer_id AND t.is_current = TRUE
WHERE t.customer_id IS NULL
""")

spark.sql(f"""
MERGE INTO {target} t
USING type2_changes c
ON t.customer_id = c.customer_id AND t.is_current = TRUE
WHEN MATCHED THEN UPDATE SET
    t.effective_to = DATE_SUB(c.signup_date, 1),
    t.is_current = FALSE
""")

spark.sql(f"""
INSERT INTO {target}
SELECT
    c.customer_id, c.full_name, c.email, c.phone, c.city, c.tier,
    c.signup_date AS effective_from, NULL AS effective_to, TRUE AS is_current
FROM type2_changes c
""")

spark.sql(f"""
MERGE INTO {target} t
USING (
    SELECT s.customer_id, s.full_name, s.email, s.phone
    FROM (
        SELECT DISTINCT customer_id, full_name, email, phone, city, tier, signup_date
        FROM {source}
        WHERE customer_id IS NOT NULL AND signup_date IS NOT NULL
    ) s
    JOIN {target} t
        ON s.customer_id = t.customer_id AND t.is_current = TRUE
    WHERE s.signup_date > t.effective_from
      AND s.city <=> t.city AND s.tier <=> t.tier
      AND (NOT (s.full_name <=> t.full_name) OR NOT (s.email <=> t.email) OR NOT (s.phone <=> t.phone))
) c
ON t.customer_id = c.customer_id AND t.is_current = TRUE
WHEN MATCHED THEN UPDATE SET
    t.full_name = c.full_name,
    t.email = c.email,
    t.phone = c.phone
""")

DataFrame[num_affected_rows: bigint, num_updated_rows: bigint, num_deleted_rows: bigint, num_inserted_rows: bigint]

In [0]:
%sql
select * from fk_ce1366.silver.customers_scd_t2

customer_id,full_name,email,phone,city,tier,effective_from,effective_to,is_current
C0001,Kiara Mehta,kiara.mehta1@outlook.com,9967103366,Jaipur,Basic,2026-03-01,null,true
C0002,Aditya Gupta,aditya.gupta2@gmail.com,+919839814388,Delhi,Premium,2026-03-01,null,true
C0003,Ananya Shekhawat,ananya.shekhawat3@yahoo.in,919935231776,Mumbai,Plus,2026-03-01,null,true
C0004,Yash Iyer,yash.iyer4@gmail.com,9014941883,Bengaluru,Premium,2026-03-01,null,true
C0005,Tara Nair,tara.nair5@outlook.com,9445672481,Pune,Plus,2026-03-01,null,true
C0006,Diya Joshi,diya.joshi6@yahoo.in,9236469229,Hyderabad,Basic,2026-03-01,null,true
C0007,Nikhil Nair,nikhil.nair7@yahoo.in,9817804830,Jaipur,Plus,2026-03-01,null,true
C0008,Varun Gupta,varun.gupta8@yahoo.in,09949018941,Delhi,Basic,2026-03-01,null,true
C0009,Kiara Gupta,kiara.gupta9@outlook.com,9064120097,Mumbai,Basic,2026-03-01,null,true
C0010,Rahul Malhotra,rahul.malhotra10@outlook.com,+919564035207,Bengaluru,Basic,2026-03-01,null,true


In [0]:
%sql
select count(*) from fk_ce1366.bronze.customers

count(*)
403
